In [18]:
import dask.dataframe as ddf
import pandas as pd

import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from itertools import product
import optuna

features = ddf.read_parquet(
    '/Users/leo/Documents/gpl/featurization_competition/featurization-urap-remote-execution/data_for_bucket/input_data/togo/togo_features_2018_10/features/*.parquet'
).compute()
holdout_subscribers = set(pd.read_csv(
    '/Users/leo/Documents/gpl/featurization_competition/featurization-urap-remote-execution/data_for_bucket/input_data/togo/togo_full_2018_10/hold_out_subscribers.csv'
).subscriber_id.values)
un_held_out_features = features[~features.name.isin(holdout_subscribers)]
consumption = pd.read_csv(
    '/Users/leo/Documents/gpl/featurization_competition/featurization-urap-remote-execution/data_for_bucket/input_data/togo/togo_full_2018_10/combined_real_consumption.csv'
)
merged = un_held_out_features.merge(consumption, left_on='name', right_on='phone_number').drop(
    columns=['name', 'hhid']
)


optuna.logging.set_verbosity(optuna.logging.WARNING)

feature_cols = [c for c in merged.columns if c not in ['phone_number', 'consumption']]
X = merged[feature_cols].values
y = merged['consumption'].values

N_TRIALS = 100

outer_cv = KFold(n_splits=2, shuffle=True, random_state=42)
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

oos_predictions = np.zeros(len(y))

search_bounds = {
    'learning_rate': (0.01, 0.3),
    'max_depth':     (1, 8),
    'num_leaves':    (5, 60),
    'min_child_samples': (20, 500),
}

def make_objective(X_outer_train, y_outer_train):
    def objective(trial):
        params = {
            'n_estimators':      1000,
            'learning_rate':     trial.suggest_float('learning_rate', *search_bounds['learning_rate'], log=True),
            'max_depth':         trial.suggest_int('max_depth', *search_bounds['max_depth']),
            'num_leaves':        trial.suggest_int('num_leaves', *search_bounds['num_leaves']),
            'min_child_samples': trial.suggest_int('min_child_samples', *search_bounds['min_child_samples'], log=True),
            'random_state': 42,
            'verbose': -1,
        }

        inner_scores = []
        for inner_train_idx, inner_val_idx in inner_cv.split(X_outer_train):
            X_inner_train = X_outer_train[inner_train_idx]
            y_inner_train = y_outer_train[inner_train_idx]
            X_inner_val   = X_outer_train[inner_val_idx]
            y_inner_val   = y_outer_train[inner_val_idx]

            model = lgb.LGBMRegressor(**params)
            model.fit(
                X_inner_train, y_inner_train,
                eval_set=[(X_inner_val, y_inner_val)],
                callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
            )
            inner_scores.append(r2_score(y_inner_val, model.predict(X_inner_val)))

        return np.mean(inner_scores)
    return objective

for outer_fold, (train_idx, test_idx) in enumerate(outer_cv.split(X)):
    X_outer_train, X_outer_test = X[train_idx], X[test_idx]
    y_outer_train, y_outer_test = y[train_idx], y[test_idx]

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(make_objective(X_outer_train, y_outer_train), n_trials=N_TRIALS)

    best = study.best_params
    print(f"\nOuter fold {outer_fold + 1}: best inner CV R² = {study.best_value:.4f}")
    print(f"  Best params: {best}")

    # Edge diagnostic
    edge_params = []
    for k, (lo, hi) in search_bounds.items():
        v = best[k]
        at_lo = (v == lo) if isinstance(v, int) else (v <= lo * 1.01)
        at_hi = (v == hi) if isinstance(v, int) else (v >= hi * 0.99)
        if at_lo or at_hi:
            edge_params.append(f"{k}={v} ({'low' if at_lo else 'high'} boundary)")
    if edge_params:
        print(f"  WARNING: params on search boundary: {edge_params}")
    else:
        print(f"  All params interior to search space.")

    # Refit on full outer training fold
    final_model = lgb.LGBMRegressor(**best, n_estimators=1000, random_state=42, verbose=-1)
    final_model.fit(X_outer_train, y_outer_train)

    oos_predictions[test_idx] = final_model.predict(X_outer_test)
    print(f"Outer fold {outer_fold + 1} OOS R²: {r2_score(y_outer_test, oos_predictions[test_idx]):.4f}")

print(f"\nOverall OOS R² (combined): {r2_score(y, oos_predictions):.4f}")


In [24]:
predictions_df = merged[['phone_number']].copy()
predictions_df['predicted_consumption'] = oos_predictions
predictions_df.to_csv('misc/un_held_out_predictions.csv', index=False)